# CAP5636 — Project: Noise-Robust Food Image Classification
**Noise-Robust Fine-Grained Food Image Classification via Transfer Learning and Label-Smoothing Regularization**

Julian McKinley · Cristhian Cruz

---

## Learning Outcomes
By completing this notebook you will be able to:
- Understand why fine-grained food classification is hard (inter-class visual similarity + label noise)
- Apply **transfer learning** with two-phase fine-tuning (warm-up → top-block unfreeze)
- Implement and compare four noise-robust training strategies: standard CE, label smoothing, SCE, CutMix
- Quantitatively compare configurations using Top-1 / Top-5 accuracy, with a proper train/val/test split and a bootstrap significance test
- Qualitatively interpret model decisions using **Grad-CAM** and confusion matrices

## Notes on this version
A few methodology fixes relative to an earlier draft of this notebook, worth calling out explicitly:
- **No leakage between model selection and reporting.** The earlier draft validated against the official test set every epoch and picked "best" based on it — that biases the final number. Here, a stratified 10% split is carved out of the **train** set for per-epoch monitoring/checkpoint selection; the official test set is touched **exactly once per configuration**, at the end.
- **Single, fixed configuration** — no fast/full toggle. Backbone is **EfficientNetV2-S**, trained at **224×224** for all backbones (see the note in Section 1 on why, given EfficientNetV2-S's native resolution is 384×384).
- **Best-checkpoint reload.** Each experiment reloads its best validation checkpoint before final evaluation, rather than reporting whatever the last training epoch happened to leave in memory.
- **Early stopping + per-epoch checkpointing to disk**, so a Kaggle session timeout doesn't lose finished experiments.
- Mixed precision, `cudnn.benchmark`, and `nn.DataParallel` (to use both Kaggle GPUs) are enabled for speed.


## 1) Setup & Configuration

In [ ]:
# ── 1a) Imports ──────────────────────────────────────────────────────────────
import os, json, time, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import autocast, GradScaler   # unified CUDA/CPU AMP API (torch>=2.0)
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.models as tv_models
import torchvision.transforms as transforms
from torchvision.datasets import Food101
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
warnings.filterwarnings("ignore")
print("Imports OK")


In [ ]:
# ── 1b) Configuration ─────────────────────────────────────────────────────────
DATA_DIR   = "./data"
OUTPUT_DIR = "./outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PRIMARY_BACKBONE  = "efficientnet_v2_s"   # noise-robust experiments (E1-E4)
BASELINE_BACKBONE = "resnet50"            # architecture baseline (E0)
N_CLASSES         = 101

IMG_SIZE      = 224     # NOTE: EfficientNetV2-S's pretrained weights are natively 384x384
                         # (torchvision: resize=384, crop=384, bilinear). We deliberately
                         # train at 224 across BOTH backbones to fit a 12-hour Kaggle budget.
                         # This isn't just a compute shortcut: the EfficientNetV2 paper
                         # itself found that training at a smaller resolution than the
                         # eval resolution ("FixRes"-style) speeds up training substantially
                         # with a small, often slightly *positive*, accuracy effect
                         # (Tan & Le, 2021). See Section 10 for the full discussion.
BATCH_SIZE    = 64
LR            = 1e-4
WEIGHT_DECAY  = 1e-2

VAL_FRACTION       = 0.10   # carved out of the TRAIN split only; test set is never used for this
UNFREEZE_BLOCKS    = 3      # top EfficientNet stage-groups to unfreeze at fine-tune phase
WARMUP_EPOCHS      = 3
FINETUNE_EPOCHS    = 15
EARLY_STOP_PATIENCE = 4     # stop fine-tuning if val_top1 hasn't improved in this many epochs
NUM_WORKERS         = 4

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

print(f"Primary backbone : {PRIMARY_BACKBONE}")
print(f"Baseline backbone: {BASELINE_BACKBONE}")
print(f"Image size       : {IMG_SIZE}x{IMG_SIZE}")
print(f"Warmup epochs    : {WARMUP_EPOCHS}  |  Fine-tune epochs (max): {FINETUNE_EPOCHS}  |  Early-stop patience: {EARLY_STOP_PATIENCE}")


In [ ]:
# ── 1c) Device, precision, and reproducibility info ───────────────────────────
device      = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS      = torch.cuda.device_count()
AMP_DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"
AMP_ENABLED = torch.cuda.is_available()

torch.backends.cudnn.benchmark = True   # safe here since IMG_SIZE is fixed for every batch

print(f"torch        : {torch.__version__}")
print(f"torchvision  : {torchvision.__version__}")
print(f"Device       : {device}")
print(f"Visible GPUs : {N_GPUS}")
print(f"AMP enabled  : {AMP_ENABLED}")
if N_GPUS > 1:
    print(f"nn.DataParallel will be used across all {N_GPUS} visible GPUs during training.")
elif N_GPUS == 1:
    print("Only 1 GPU visible - if Kaggle's second GPU isn't showing up, check the accelerator setting.")


## 2) The Food-101 Dataset

**Food-101** (Bossard, Guillaumin & Van Gool, ECCV 2014) contains 101,000 images across 101 food categories:

| Split | Images | Notes |
|---|---|---|
| Train | 75,750 | Collected automatically from Foodspotting — **contains label noise** |
| Test | 25,250 | Manually verified — **clean** |

The original paper describes the training-set noise specifically as coming "mostly in the form of intense
colors and sometimes wrong labels" — i.e. it's not purely a mislabeling problem, some of it is closer to
visual/exposure noise. All images are rescaled so the longer side is at most 512px (not to a fixed
resolution), which the EDA below verifies empirically.

**A note on a similarly-named dataset:** don't confuse this with **Food-101N** (Lee et al., CVPR 2018), a
separate, much larger (~310k image) and noisier dataset built later with the same 101 classes, with an
*estimated* noisy-label accuracy of ~80%. That 80% figure is specific to Food-101N — the original Food-101
paper never quantifies its own noise rate numerically, and we don't either; we treat "noisy" qualitatively,
as the paper does.

> **Dataset:** https://www.kaggle.com/datasets/kmader/food41
> torchvision downloads it automatically on first run.

**Methodology note:** to avoid tuning on the test set, we carve a stratified 10% validation split out of
the *train* set for per-epoch monitoring and checkpoint selection. The official test set is only touched
once per configuration, at the very end, for the number that goes in the results table.


In [ ]:
# ── 2a) Transforms ─────────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
# Inverse of the normalization above, used whenever we need to display a tensor as an image.
INV_NORM = transforms.Normalize(
    mean=[-m / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1.0 / s for s in IMAGENET_STD],
)
print("Transforms defined.")


In [ ]:
# ── 2b) Load Food-101 and build a leakage-free train / val / test split ────────
def get_labels_array(dataset):
    '''Fetch integer labels for a torchvision Food101 dataset.

    Uses the (undocumented, underscore-prefixed) `_labels` attribute when
    available, since it's O(1) - it's already built from the split's metadata
    JSON at __init__ time, no image I/O needed. Falls back to a slower
    per-item scan otherwise, so this keeps working even if a future
    torchvision release renames/removes the private attribute.
    '''
    if hasattr(dataset, "_labels"):
        return np.array(dataset._labels)
    print("  [warn] `_labels` not found on dataset; scanning items instead (slower)...")
    return np.array([dataset[i][1] for i in range(len(dataset))])


print("Downloading / loading Food-101 (first run only)...")
# Two separate instances of the *same* train split: one with augmentation (used for
# gradient updates), one with eval-style preprocessing (used for the held-out val subset).
# Both are built from the same metadata JSON in a fixed order, so indices line up 1:1
# across instances - see the assertion below.
train_full_aug  = Food101(root=DATA_DIR, split="train", transform=train_transform, download=True)
train_full_eval = Food101(root=DATA_DIR, split="train", transform=eval_transform,  download=False)
test_ds         = Food101(root=DATA_DIR, split="test",  transform=eval_transform,  download=True)

CLASS_NAMES = train_full_aug.classes
assert CLASS_NAMES == train_full_eval.classes == test_ds.classes
assert len(train_full_aug) == len(train_full_eval)

labels = get_labels_array(train_full_aug)
all_idx = np.arange(len(train_full_aug))
train_idx, val_idx = train_test_split(
    all_idx, test_size=VAL_FRACTION, stratify=labels, random_state=SEED
)

train_ds = Subset(train_full_aug,  train_idx)   # augmented -> used for gradient updates
val_ds   = Subset(train_full_eval, val_idx)     # eval-style transform -> used for model selection

print(f"Classes           : {len(CLASS_NAMES)}")
print(f"Train (grad. upd.): {len(train_ds):,}")
print(f"Val (model sel.)  : {len(val_ds):,}")
print(f"Test (held out)   : {len(test_ds):,}")


In [ ]:
# ── 2c) DataLoaders ────────────────────────────────────────────────────────────
_loader_kwargs = dict(
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0),
    prefetch_factor=4 if NUM_WORKERS > 0 else None,
)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  **_loader_kwargs)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, **_loader_kwargs)
print(f"train_loader: {len(train_loader)} batches | val_loader: {len(val_loader)} batches | test_loader: {len(test_loader)} batches")


## 3) Exploratory Data Analysis

Before modeling, we look at what actually makes this dataset hard: fine-grained visual similarity between
classes, real-world image variability, and the noise the proposal is built around addressing.

In [ ]:
# ── 3a) Class balance across splits ────────────────────────────────────────────
raw_view_ds = Food101(root=DATA_DIR, split="train", transform=None, download=False)
assert raw_view_ds.classes == CLASS_NAMES  # same ordering as train_full_aug / train_full_eval

train_labels_split = labels[train_idx]
val_labels_split   = labels[val_idx]
test_labels        = get_labels_array(test_ds)

train_counts = np.bincount(train_labels_split, minlength=N_CLASSES)
val_counts   = np.bincount(val_labels_split,   minlength=N_CLASSES)
test_counts  = np.bincount(test_labels,        minlength=N_CLASSES)

print(f"Train per-class : min={train_counts.min()}  max={train_counts.max()}  mean={train_counts.mean():.1f}")
print(f"Val per-class   : min={val_counts.min()}  max={val_counts.max()}  mean={val_counts.mean():.1f}")
print(f"Test per-class  : min={test_counts.min()}  max={test_counts.max()}  mean={test_counts.mean():.1f}  "
      f"(Bossard et al. specify exactly 250/class for test)")

fig, ax = plt.subplots(figsize=(16, 4))
x = np.arange(N_CLASSES)
ax.bar(x, train_counts, label="train (post-split)", color="steelblue")
ax.bar(x, val_counts, bottom=train_counts, label="val (held out of train)", color="orange")
ax.set_xlabel("Class index"); ax.set_ylabel("# images")
ax.set_title("Food-101 class balance across splits"); ax.legend()
plt.tight_layout(); plt.show()


In [ ]:
# ── 3b) Raw image dimensions & aspect ratio ────────────────────────────────────
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(raw_view_ds), size=500, replace=False)

widths, heights = [], []
for idx in sample_idx:
    img, _ = raw_view_ds[int(idx)]
    w, h = img.size
    widths.append(w); heights.append(h)
widths, heights = np.array(widths), np.array(heights)
max_side = np.maximum(widths, heights)

print(f"Max side length (n={len(sample_idx)} sample): min={max_side.min()}  max={max_side.max()}  mean={max_side.mean():.0f}")
print("Bossard et al. (2014) rescale all images so the longer side is <=512px - confirmed above.")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(widths, bins=30, color="steelblue");  axes[0].set_title("Width (px)")
axes[1].hist(heights, bins=30, color="steelblue"); axes[1].set_title("Height (px)")
axes[2].hist(widths / heights, bins=30, color="steelblue"); axes[2].set_title("Aspect ratio (w/h)")
plt.tight_layout(); plt.show()


In [ ]:
# ── 3c) Brightness / color intensity distribution ─────────────────────────────
# Motivates ColorJitter as an augmentation choice, given the "intense colors" noise
# Bossard et al. describe in the training split.
brightness = []
for idx in sample_idx[:200]:
    img, _ = raw_view_ds[int(idx)]
    arr = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0
    brightness.append(arr.mean())
brightness = np.array(brightness)

plt.figure(figsize=(6, 4))
plt.hist(brightness, bins=30, color="steelblue")
plt.xlabel("Mean pixel brightness (0-1)"); plt.title("Brightness distribution (sampled training images)")
plt.tight_layout(); plt.show()
print(f"Mean brightness: {brightness.mean():.3f}  std: {brightness.std():.3f}")


In [ ]:
# ── 3d) Visually confusable class pairs ────────────────────────────────────────
# Straight from the proposal's own framing of why this dataset is hard.
CONFUSABLE_PAIRS = [
    ("chocolate_cake", "chocolate_mousse"),
    ("beef_tartare", "pulled_pork_sandwich"),
    ("filet_mignon", "prime_rib"),
]

fig, axes = plt.subplots(len(CONFUSABLE_PAIRS), 6, figsize=(15, 3 * len(CONFUSABLE_PAIRS)))
rng2 = np.random.default_rng(SEED + 1)
for row, (name_a, name_b) in enumerate(CONFUSABLE_PAIRS):
    for col, name in enumerate([name_a] * 3 + [name_b] * 3):
        cls_idx = train_full_aug.class_to_idx[name]
        cls_pool = np.where(labels == cls_idx)[0]
        pick = rng2.choice(cls_pool)
        img, _ = raw_view_ds[int(pick)]
        axes[row, col].imshow(img)
        axes[row, col].axis("off")
    axes[row, 0].set_title(name_a.replace("_", " "), fontsize=10, loc="left")
    axes[row, 3].set_title(name_b.replace("_", " "), fontsize=10, loc="left")
plt.suptitle("Visually confusable class pairs (the fine-grained challenge)", y=1.01)
plt.tight_layout(); plt.show()


In [ ]:
# ── 3e) Sample training image grid ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 8, figsize=(15, 4.5))
grid_idx = rng.choice(len(raw_view_ds), 16, replace=False)
for ax, idx in zip(axes.flat, grid_idx):
    img, label = raw_view_ds[int(idx)]
    ax.imshow(img)
    ax.set_title(CLASS_NAMES[label].replace("_", " "), fontsize=7, fontweight="bold")
    ax.axis("off")
plt.suptitle("Sample Training Images - Food-101", y=1.02)
plt.tight_layout(); plt.show()


In [ ]:
# ── 3f) Augmentation preview ────────────────────────────────────────────────────
# Same source image, put through train_transform several times, to sanity-check
# what the model actually sees during training (RandomResizedCrop + flip + jitter).
preview_img, preview_label = raw_view_ds[int(sample_idx[0])]
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
axes[0].imshow(preview_img); axes[0].set_title("Original"); axes[0].axis("off")
for i in range(1, 5):
    aug_tensor = train_transform(preview_img)
    aug_img = INV_NORM(aug_tensor).permute(1, 2, 0).clamp(0, 1).numpy()
    axes[i].imshow(aug_img); axes[i].set_title(f"Augmented #{i}"); axes[i].axis("off")
plt.suptitle(f"Augmentation preview - {CLASS_NAMES[preview_label].replace('_', ' ')}")
plt.tight_layout(); plt.show()


In [ ]:
# ── 3g) EDA summary ─────────────────────────────────────────────────────────────
summary_rows = [
    ("Train images (full, before val split)", f"{len(train_full_aug):,}"),
    ("Train images (used for gradient updates)", f"{len(train_ds):,}"),
    ("Val images (held out of train, for model selection)", f"{len(val_ds):,}"),
    ("Test images (official clean holdout, touched once per config)", f"{len(test_ds):,}"),
    ("Classes", f"{N_CLASSES}"),
    ("Images per class (test, official)", "250"),
    ("Sampled image max-side range (px)", f"{max_side.min()}-{max_side.max()}"),
    ("Normalization mean / std", f"{IMAGENET_MEAN} / {IMAGENET_STD}"),
    ("Training resolution", f"{IMG_SIZE}x{IMG_SIZE}"),
]
pd.DataFrame(summary_rows, columns=["Metric", "Value"])


## 4) Noise-Robust Loss Functions

Standard cross-entropy assigns probability 1.0 to whatever label the training image carries. When labels
are noisy, this forces the model to memorize wrong annotations. We compare four strategies:

| # | Strategy | Key idea |
|---|---|---|
| CE | Standard cross-entropy | Baseline; overconfident on noisy labels |
| LS | Label smoothing (ε = 0.1) | Distributes 0.1 probability mass across all classes, preventing overconfidence |
| SCE | Symmetric Cross Entropy (Wang et al., 2019) | Adds a reverse-CE term that penalises confident wrong predictions |
| CM | CutMix (Yun et al., 2019) | Mixes image patches and labels from two samples; acts as a strong augmentation and implicit regulariser |


In [ ]:
# ── 4a) Symmetric Cross Entropy ─────────────────────────────────────────────────
class SymmetricCrossEntropyLoss(nn.Module):
    '''SCE = alpha * CE(p, q) + beta * RCE(p, q). Default alpha=0.1, beta=1.0 (Wang et al., ICCV 2019).'''
    def __init__(self, alpha=0.1, beta=1.0):
        super().__init__()
        self.alpha, self.beta = alpha, beta

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets)
        pred = F.softmax(logits, dim=1).clamp(1e-7, 1.0)
        oh = torch.zeros_like(pred).scatter_(1, targets.unsqueeze(1), 1.0).clamp(1e-4, 1.0)
        rce = -(pred * oh.log()).sum(1).mean()
        return self.alpha * ce + self.beta * rce


# ── 4b) CutMix helpers ───────────────────────────────────────────────────────────
def _rand_bbox(W, H, lam):
    cut_w, cut_h = int(W * (1 - lam) ** 0.5), int(H * (1 - lam) ** 0.5)
    cx, cy = np.random.randint(W), np.random.randint(H)
    x1, y1 = max(cx - cut_w // 2, 0), max(cy - cut_h // 2, 0)
    x2, y2 = min(cx + cut_w // 2, W), min(cy + cut_h // 2, H)
    return x1, y1, x2, y2

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    ya, yb = y, y[idx]
    x1, y1, x2, y2 = _rand_bbox(x.size(3), x.size(2), lam)
    x_mix = x.clone(); x_mix[:, :, y1:y2, x1:x2] = x[idx, :, y1:y2, x1:x2]
    lam = 1 - (x2 - x1) * (y2 - y1) / (x.size(2) * x.size(3))
    return x_mix, ya, yb, lam

def cutmix_loss(criterion, out, ya, yb, lam):
    return lam * criterion(out, ya) + (1 - lam) * criterion(out, yb)


# ── 4c) Loss factory ─────────────────────────────────────────────────────────────
def get_loss_fn(name, n_classes=N_CLASSES):
    if name == "ce":              return nn.CrossEntropyLoss()
    if name == "label_smoothing": return nn.CrossEntropyLoss(label_smoothing=0.1)
    if name == "sce":             return SymmetricCrossEntropyLoss(alpha=0.1, beta=1.0)
    if name == "cutmix":          return nn.CrossEntropyLoss()   # mixing happens in the train loop
    raise ValueError(name)

print("Loss functions defined.")


## 5) Model Factory & Transfer Learning Setup

### Transfer Learning Strategy

We use backbones pretrained on ImageNet-1K and adapt them to Food-101 in **two phases**:

```
Phase 1 — Warm-up (head only)
  All backbone weights frozen -> only the new classification head learns.
  Uses a higher LR (10x) for fast head initialisation.

Phase 2 — Fine-tuning (top blocks + head)
  Unfreeze the top 3 stage-groups; keep early layers frozen.
  Early layers capture generic edge/texture features - no need to disturb them.
```

| Backbone | Params | ImageNet Top-1 | Role |
|---|---|---|---|
| ResNet-50 | 25.6M | 76.1% | Architecture baseline |
| EfficientNetV2-S | 21.5M | 84.2% | Primary backbone |

(ResNet-50 and EfficientNetV2-S numbers are torchvision's `IMAGENET1K_V1` weights metadata.)


In [ ]:
# ── 5a) Model factory ────────────────────────────────────────────────────────────
def get_model(backbone, n_classes=N_CLASSES, pretrained=True):
    if backbone == "efficientnet_v2_s":
        w = tv_models.EfficientNet_V2_S_Weights.IMAGENET1K_V1 if pretrained else None
        mdl = tv_models.efficientnet_v2_s(weights=w)
        mdl.classifier[1] = nn.Linear(mdl.classifier[1].in_features, n_classes)
    elif backbone == "resnet50":
        w = tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None
        mdl = tv_models.resnet50(weights=w)
        mdl.fc = nn.Linear(mdl.fc.in_features, n_classes)
    else:
        raise ValueError(f"Unknown backbone: {backbone}")
    return mdl

def count_params(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def freeze_backbone(model, backbone):
    '''Freeze everything except the classification head (warm-up phase).'''
    if backbone.startswith("efficientnet"):
        for p in model.features.parameters():   p.requires_grad = False
        for p in model.classifier.parameters(): p.requires_grad = True
    else:  # resnet50
        for n, p in model.named_parameters():
            p.requires_grad = ("fc" in n)
    _, tr = count_params(model)
    print(f"    [freeze] trainable params: {tr:,}")

def unfreeze_top(model, backbone, n_blocks=UNFREEZE_BLOCKS):
    '''Unfreeze top n_blocks + head for the fine-tuning phase.'''
    if backbone.startswith("efficientnet"):
        for p in model.classifier.parameters(): p.requires_grad = True
        blocks = list(model.features)
        for i, blk in enumerate(blocks):
            trainable = i >= len(blocks) - n_blocks
            for p in blk.parameters(): p.requires_grad = trainable
    else:
        for p in model.parameters(): p.requires_grad = True
    _, tr = count_params(model)
    print(f"    [unfreeze top-{n_blocks}] trainable params: {tr:,}")

print("Model factory defined.")


## 6) Training Utilities

Mixed precision, `nn.DataParallel` (when >1 GPU is visible), and early stopping are all handled here.

In [ ]:
# ── 6a) Metrics & early stopping ────────────────────────────────────────────────
def topk_acc(output, target, topk=(1, 5)):
    '''Return Top-k accuracy (%) as a list.'''
    with torch.no_grad():
        maxk = max(topk)
        B = target.size(0)
        _, pred = output.topk(maxk, dim=1)
        pred = pred.t()
        correct = pred.eq(target.view(1, -1).expand_as(pred))
        return [correct[:k].reshape(-1).float().sum().mul_(100.0 / B).item() for k in topk]

class EarlyStopper:
    '''Stops fine-tuning once val_top1 hasn't improved in `patience` epochs.'''
    def __init__(self, patience=EARLY_STOP_PATIENCE, min_delta=0.0):
        self.patience, self.min_delta = patience, min_delta
        self.best = -float("inf")
        self.counter = 0

    def step(self, value):
        if value > self.best + self.min_delta:
            self.best, self.counter = value, 0
        else:
            self.counter += 1
        return self.counter >= self.patience

def maybe_parallel(model):
    '''Wrap in nn.DataParallel when more than one GPU is visible - this is what
    actually puts Kaggle's second GPU to work; the earlier draft never did this.'''
    return nn.DataParallel(model) if torch.cuda.device_count() > 1 else model

print("Metrics & early stopping utilities defined.")


In [ ]:
# ── 6b) Train / eval loops (AMP-enabled) ────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, scaler, use_cutmix=False):
    model.train()
    ls = t1 = t5 = n = 0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            if use_cutmix:
                mixed_imgs, ya, yb, lam = cutmix_data(imgs, lbls)
                out  = model(mixed_imgs)
                loss = cutmix_loss(criterion, out, ya, yb, lam)
            else:
                out  = model(imgs)
                loss = criterion(out, lbls)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        a1, a5 = topk_acc(out.detach(), lbls, topk=(1, 5))
        ls += loss.item(); t1 += a1; t5 += a5; n += 1
    return ls / n, t1 / n, t5 / n

@torch.no_grad()
def eval_one_epoch(model, loader, criterion):
    model.eval()
    ls = t1 = t5 = n = 0
    for imgs, lbls in loader:
        imgs = imgs.to(device, non_blocking=True)
        lbls = lbls.to(device, non_blocking=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            out  = model(imgs)
            loss = criterion(out, lbls)
        a1, a5 = topk_acc(out, lbls, topk=(1, 5))
        ls += loss.item(); t1 += a1; t5 += a5; n += 1
    return ls / n, t1 / n, t5 / n

print("Training / eval loops defined.")


In [ ]:
# ── 6c) Full two-phase training loop ────────────────────────────────────────────
def run_experiment(backbone, loss_name, label="", n_classes=N_CLASSES):
    '''Two-phase training: warm-up (head only) -> fine-tune (top blocks), with
    early stopping, per-epoch checkpointing to disk, best-checkpoint reload, and
    a single held-out TEST evaluation at the end.'''
    mdl  = get_model(backbone=backbone, n_classes=n_classes).to(device)
    crit = get_loss_fn(loss_name, n_classes)
    use_cm = (loss_name == "cutmix")
    run_id = f"{backbone}_{loss_name}"
    print(f"\n{'=' * 70}\n  {label or run_id}\n{'=' * 70}")

    hist = {k: [] for k in ["phase", "tr_loss", "tr_top1", "tr_top5", "vl_loss", "vl_top1", "vl_top5", "epoch_seconds"]}
    best_top1, best_ckpt = 0.0, os.path.join(OUTPUT_DIR, f"{run_id}_best.pth")
    hist_path = os.path.join(OUTPUT_DIR, f"{run_id}_history.json")

    def _log_epoch(phase, tr, vl, dt):
        hist["phase"].append(phase)
        hist["tr_loss"].append(tr[0]); hist["tr_top1"].append(tr[1]); hist["tr_top5"].append(tr[2])
        hist["vl_loss"].append(vl[0]); hist["vl_top1"].append(vl[1]); hist["vl_top5"].append(vl[2])
        hist["epoch_seconds"].append(dt)
        with open(hist_path, "w") as f:   # persisted every epoch - a timeout won't lose finished runs
            json.dump(hist, f, indent=2)

    # ── Phase 1: warm-up ────────────────────────────────────────────
    if backbone.startswith("efficientnet") and WARMUP_EPOCHS > 0:
        print(f"  Phase 1 - Warm-up ({WARMUP_EPOCHS} epochs, head only)")
        freeze_backbone(mdl, backbone)
        opt = optim.AdamW(filter(lambda p: p.requires_grad, mdl.parameters()), lr=LR * 10, weight_decay=WEIGHT_DECAY)
        sched = CosineAnnealingLR(opt, T_max=WARMUP_EPOCHS)
        scaler = GradScaler(AMP_DEVICE, enabled=AMP_ENABLED)
        mdl_fwd = maybe_parallel(mdl)
        for ep in range(WARMUP_EPOCHS):
            t0 = time.time()
            tr = train_one_epoch(mdl_fwd, train_loader, opt, crit, scaler, use_cm)
            vl = eval_one_epoch(mdl_fwd, val_loader, crit)
            sched.step()
            _log_epoch("warmup", tr, vl, time.time() - t0)
            print(f"    WU {ep + 1:02d}/{WARMUP_EPOCHS}  train_loss={tr[0]:.3f}  "
                  f"val_top1={vl[1]:.1f}%  val_top5={vl[2]:.1f}%  ({time.time() - t0:.0f}s)")

    # ── Phase 2: fine-tune ──────────────────────────────────────────
    print(f"  Phase 2 - Fine-tuning (up to {FINETUNE_EPOCHS} epochs, early-stop patience={EARLY_STOP_PATIENCE})")
    unfreeze_top(mdl, backbone)
    opt = optim.AdamW(filter(lambda p: p.requires_grad, mdl.parameters()), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = CosineAnnealingLR(opt, T_max=FINETUNE_EPOCHS)
    scaler = GradScaler(AMP_DEVICE, enabled=AMP_ENABLED)
    mdl_fwd = maybe_parallel(mdl)
    stopper = EarlyStopper(patience=EARLY_STOP_PATIENCE)

    for ep in range(FINETUNE_EPOCHS):
        t0 = time.time()
        tr = train_one_epoch(mdl_fwd, train_loader, opt, crit, scaler, use_cm)
        vl = eval_one_epoch(mdl_fwd, val_loader, crit)
        sched.step()
        _log_epoch("finetune", tr, vl, time.time() - t0)
        star = ""
        if vl[1] > best_top1:
            best_top1 = vl[1]
            torch.save(mdl.state_dict(), best_ckpt)
            star = " *"
        print(f"    EP {ep + 1:02d}/{FINETUNE_EPOCHS}  train_loss={tr[0]:.3f}  train_top1={tr[1]:.1f}%  "
              f"val_top1={vl[1]:.1f}%  val_top5={vl[2]:.1f}%  ({time.time() - t0:.0f}s){star}")
        if stopper.step(vl[1]):
            print(f"    Early stopping (no val_top1 improvement in {EARLY_STOP_PATIENCE} epochs).")
            break

    # Reload the *best* checkpoint - NOT just whatever the last epoch left in memory.
    mdl.load_state_dict(torch.load(best_ckpt, map_location=device))
    mdl.eval()

    # Touch the official TEST set exactly once, for the number that goes in the results table.
    test_loss, test_top1, test_top5 = eval_one_epoch(maybe_parallel(mdl), test_loader, crit)
    print(f"  Best val Top-1: {best_top1:.2f}%   |   Held-out TEST Top-1: {test_top1:.2f}%  Top-5: {test_top5:.2f}%")

    return {
        "label": label or run_id, "run_id": run_id, "history": hist,
        "best_val_top1": best_top1,
        "test_loss": test_loss, "test_top1": test_top1, "test_top5": test_top5,
        "model": mdl, "checkpoint": best_ckpt,
        "backbone": backbone, "loss": loss_name,
    }

print("run_experiment() defined - ready to train.")


## 7) Experiments

Five configurations, run in sequence. Results accumulate in `RESULTS` for side-by-side comparison in Section 8.

| ID | Backbone | Loss | Purpose |
|---|---|---|---|
| E0 | ResNet-50 | CE | Architecture baseline |
| E1 | EfficientNetV2-S | CE | Noise-method baseline |
| E2 | EfficientNetV2-S | Label Smoothing | Noise-robust via soft targets |
| E3 | EfficientNetV2-S | SCE | Noise-robust via symmetric loss |
| E4 | EfficientNetV2-S | CutMix | Noise-robust via patch mixing |


In [ ]:
RESULTS = {}  # accumulates all experiment results


### E0 — Architecture Baseline: ResNet-50 + CE

**Goal:** establish how much the backbone matters *independently of the loss function*. ResNet-50 does
**not** use the warm-up phase - the entire network is unfrozen from the start.

In [ ]:
RESULTS["E0"] = run_experiment(
    backbone  = BASELINE_BACKBONE,
    loss_name = "ce",
    label     = "E0: ResNet-50 + CE (architecture baseline)",
)


### E1 — Noise-Method Baseline: EfficientNetV2-S + CE

**Goal:** isolate the effect of a stronger backbone. Identical loss to E0, but with two-phase fine-tuning.
Any accuracy improvement over E0 is attributable to the backbone, not the loss.

In [ ]:
RESULTS["E1"] = run_experiment(
    backbone  = PRIMARY_BACKBONE,
    loss_name = "ce",
    label     = f"E1: {PRIMARY_BACKBONE} + CE (noise-method baseline)",
)


### E2 — Label Smoothing (ε = 0.1)

**Goal:** test whether soft targets reduce overfitting to noisy labels. With ε=0.1 and K=101 classes, the
correct class gets probability **0.901** instead of 1.0. **Hypothesis:** E2 > E1 on the clean test set.

In [ ]:
RESULTS["E2"] = run_experiment(
    backbone  = PRIMARY_BACKBONE,
    loss_name = "label_smoothing",
    label     = f"E2: {PRIMARY_BACKBONE} + Label Smoothing (eps=0.1)",
)


### E3 — Symmetric Cross Entropy

**Goal:** test a loss *designed explicitly* for noisy-label settings. SCE (Wang et al., ICCV 2019) adds a
reverse cross-entropy term that down-weights confidently-wrong predictions. Default: α=0.1, β=1.0.

In [ ]:
RESULTS["E3"] = run_experiment(
    backbone  = PRIMARY_BACKBONE,
    loss_name = "sce",
    label     = f"E3: {PRIMARY_BACKBONE} + Symmetric CE (alpha=0.1, beta=1.0)",
)


### E4 — CutMix Augmentation

**Goal:** test data augmentation as an alternative noise-robustness mechanism. CutMix (Yun et al., ICCV
2019) mixes labels proportionally to a swapped crop's area, forcing the model to assign probability to
*two* labels simultaneously - a complementary mechanism to LS and SCE.

In [ ]:
RESULTS["E4"] = run_experiment(
    backbone  = PRIMARY_BACKBONE,
    loss_name = "cutmix",
    label     = f"E4: {PRIMARY_BACKBONE} + CutMix (alpha=1.0)",
)


## 8) Results Comparison

All numbers below are **held-out TEST set** results (touched once per configuration), not the validation
numbers used for model selection during training.

In [ ]:
# ── 8a) Summary table ────────────────────────────────────────────────────────────
rows = []
for eid, r in RESULTS.items():
    total, _ = count_params(r["model"])
    rows.append({
        "ID": eid, "Backbone": r["backbone"], "Loss": r["loss"],
        "Best Val Top-1 (%)": round(r["best_val_top1"], 2),
        "TEST Top-1 (%)": round(r["test_top1"], 2),
        "TEST Top-5 (%)": round(r["test_top5"], 2),
        "Params": f"{total / 1e6:.1f}M",
    })
results_df = pd.DataFrame(rows).sort_values("TEST Top-1 (%)", ascending=False).reset_index(drop=True)
print("=== Results Summary (ranked by held-out TEST Top-1) ===")
results_df


In [ ]:
# ── 8b) Bar chart of TEST Top-1 ──────────────────────────────────────────────────
plot_labels = [r["label"].split(": ", 1)[1] for r in RESULTS.values()]
test_accs   = [r["test_top1"] for r in RESULTS.values()]
colors      = ["#5b8db8", "#e87d4d", "#4dab6d", "#d62728", "#9467bd"][:len(RESULTS)]

plt.figure(figsize=(max(10, len(RESULTS) * 2.4), 4.5))
bars = plt.bar(plot_labels, test_accs, color=colors, edgecolor="white", width=0.6)
for bar, v in zip(bars, test_accs):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
              f"{v:.1f}%", ha="center", va="bottom", fontsize=10, fontweight="bold")
plt.ylabel("Held-out TEST Top-1 (%)"); plt.ylim(0, max(test_accs) * 1.15)
plt.title("Food-101 - TEST Top-1 by Configuration")
plt.xticks(rotation=15, ha="right"); plt.grid(axis="y", alpha=0.35)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8c) Learning curves (validation, per-epoch monitoring) ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for (eid, r), color in zip(RESULTS.items(), colors):
    vl_top1 = r["history"]["vl_top1"]
    vl_loss = r["history"]["vl_loss"]
    eps = range(1, len(vl_top1) + 1)
    lbl = r["label"].split(": ", 1)[1]
    axes[0].plot(eps, vl_top1, label=lbl, color=color, linewidth=2)
    axes[1].plot(eps, vl_loss, label=lbl, color=color, linewidth=2)
for ax, title, yl in zip(axes, ["Val Top-1 Accuracy", "Val Loss"], ["Accuracy (%)", "Loss"]):
    ax.set_title(title); ax.set_xlabel("Epoch (warm-up + fine-tune, concatenated)"); ax.set_ylabel(yl)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.35)
plt.suptitle("Food-101 - Training Curves (internal validation split, NOT test)", fontsize=12)
plt.tight_layout(); plt.show()


In [ ]:
# ── 8d) Collect held-out TEST predictions for every experiment ─────────────────
# Reused below for the confusion matrix, classification report, and bootstrap test.
@torch.no_grad()
def get_test_predictions(model):
    model.eval()
    model_fwd = maybe_parallel(model)
    all_preds, all_labels = [], []
    for imgs, lbls in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        with autocast(device_type=AMP_DEVICE, enabled=AMP_ENABLED):
            preds = model_fwd(imgs).argmax(1).cpu()
        all_preds.append(preds); all_labels.append(lbls)
    return torch.cat(all_preds).numpy(), torch.cat(all_labels).numpy()

print("Collecting held-out TEST predictions for each experiment...")
TEST_PREDS = {}
for eid, r in RESULTS.items():
    TEST_PREDS[eid] = get_test_predictions(r["model"])
    acc_check = (TEST_PREDS[eid][0] == TEST_PREDS[eid][1]).mean() * 100
    print(f"  {eid}: recomputed TEST top-1 = {acc_check:.2f}%  (reported during training: {r['test_top1']:.2f}%)")


### Statistical Significance

The proposal's evaluation plan calls for "a statistically significant margin" over the CE baseline. With
compute constraints ruling out repeated training runs per configuration, we use a **paired bootstrap** over
the fixed test set instead: resample the 25,250 test predictions with replacement many times, and look at
the distribution of the Top-1 accuracy delta between two models on each resample.

**Important caveat, stated plainly:** this CI reflects sampling uncertainty from evaluating on a
fixed-size test set — it does *not* capture training-run-to-run variance (e.g. a different random seed
producing a meaningfully different trained model). With a single trained model per configuration, that
second source of variance is a real limitation of this project, discussed further in Section 10.

In [ ]:
# ── 8e) Paired bootstrap significance test ──────────────────────────────────────
def paired_bootstrap_test(correct_a, correct_b, n_boot=10000, seed=SEED):
    '''Paired bootstrap over the test set for the Top-1 accuracy difference (a - b).'''
    rng_local = np.random.default_rng(seed)
    n = len(correct_a)
    assert n == len(correct_b)
    diffs = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng_local.integers(0, n, n)
        diffs[i] = correct_a[idx].mean() - correct_b[idx].mean()
    obs_diff = correct_a.mean() - correct_b.mean()
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    p_approx = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())  # approximate two-sided
    return {"observed_diff_pp": obs_diff * 100, "ci95_pp": (ci_lo * 100, ci_hi * 100), "p_approx": p_approx}

best_noise_eid = max(["E2", "E3", "E4"], key=lambda k: RESULTS[k]["test_top1"])
print(f"Best noise-robust config vs CE baseline: comparing {best_noise_eid} vs E1")
preds_best, labels_best = TEST_PREDS[best_noise_eid]
preds_e1,   labels_e1   = TEST_PREDS["E1"]
correct_best = (preds_best == labels_best)
correct_e1   = (preds_e1   == labels_e1)

sig_result = paired_bootstrap_test(correct_best, correct_e1)
print(f"  Observed Top-1 delta ({best_noise_eid} - E1): {sig_result['observed_diff_pp']:+.2f} pp")
print(f"  95% CI:                                      [{sig_result['ci95_pp'][0]:+.2f}, {sig_result['ci95_pp'][1]:+.2f}] pp")
print(f"  approx. two-sided p:                          {sig_result['p_approx']:.4f}")

# Architecture effect, for reference: E1 (EfficientNetV2-S) vs E0 (ResNet-50), both CE
preds_e0, labels_e0 = TEST_PREDS["E0"]
correct_e0 = (preds_e0 == labels_e0)
arch_result = paired_bootstrap_test(correct_e1, correct_e0)
print(f"\nArchitecture effect: comparing E1 vs E0 (both CE)")
print(f"  Observed Top-1 delta (E1 - E0): {arch_result['observed_diff_pp']:+.2f} pp")
print(f"  95% CI:                        [{arch_result['ci95_pp'][0]:+.2f}, {arch_result['ci95_pp'][1]:+.2f}] pp")


In [ ]:
# ── 8f) Save results to JSON ─────────────────────────────────────────────────────
summary = {}
for eid, r in RESULTS.items():
    summary[eid] = {
        "label": r["label"], "backbone": r["backbone"], "loss": r["loss"],
        "best_val_top1": r["best_val_top1"], "test_top1": r["test_top1"], "test_top5": r["test_top5"],
        "history": {k: [round(x, 4) if isinstance(x, (int, float)) else x for x in v]
                    for k, v in r["history"].items()},
    }
with open(os.path.join(OUTPUT_DIR, "notebook_results.json"), "w") as f:
    json.dump(summary, f, indent=2)
print("Results saved to", os.path.join(OUTPUT_DIR, "notebook_results.json"))


## 9) Qualitative Analysis

### 9.1 Confusion Matrix

We inspect the confusion matrix for the best-performing configuration (by TEST Top-1) to find which food
categories are most often mis-classified into each other.

In [ ]:
# ── 9.1a) Confusion matrix for the best experiment ──────────────────────────────
best_eid = max(RESULTS, key=lambda k: RESULTS[k]["test_top1"])
best_res = RESULTS[best_eid]
preds, labels_true = TEST_PREDS[best_eid]
print(f"Best experiment on held-out TEST set: {best_eid} ({best_res['label']})")
print(f"TEST Top-1: {best_res['test_top1']:.2f}%   TEST Top-5: {best_res['test_top5']:.2f}%")

TOP_N = min(15, N_CLASSES)
cm = confusion_matrix(labels_true, preds)
errors = cm.sum(1) - np.diag(cm)
top_idx = np.argsort(errors)[-TOP_N:][::-1]
cm_sub = cm[np.ix_(top_idx, top_idx)]
short = [CLASS_NAMES[i].replace("_", " ") for i in top_idx]

fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(cm_sub, annot=True, fmt="d", xticklabels=short, yticklabels=short,
            cmap="Blues", ax=ax, annot_kws={"size": 8})
ax.set_xlabel("Predicted", fontsize=11); ax.set_ylabel("True", fontsize=11)
ax.set_title(f"Confusion Matrix - Top {TOP_N} Most-Confused Classes\n{best_res['label']}")
plt.xticks(rotation=45, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout(); plt.show()


In [ ]:
# ── 9.1b) Per-class precision / recall / F1 ─────────────────────────────────────
report_dict = classification_report(labels_true, preds, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
report_df = pd.DataFrame(report_dict).T
per_class_df = report_df.iloc[:N_CLASSES]  # drop accuracy / macro avg / weighted avg rows

worst10 = per_class_df.sort_values("f1-score").head(10)
print("Hardest classes to classify (lowest F1 on the held-out test set):")
print(worst10[["precision", "recall", "f1-score", "support"]].round(3))


### 9.2 Grad-CAM: What Does the Model Actually Look At?

**Grad-CAM** (Selvaraju et al., ICCV 2017) back-propagates through the target layer and pools the
gradients to produce a class activation map. We want to see **food texture** highlighted - not background
plates, tables, or cutlery.

In [ ]:
# ── 9.2a) Grad-CAM implementation ────────────────────────────────────────────────
class GradCAM:
    def __init__(self, model, layer):
        self.model = model
        self._act = None
        self._grad = None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "_act", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "_grad", go[0].detach()))

    def __call__(self, inp, cls=None):
        self.model.eval()
        inp = inp.clone().requires_grad_(True)   # guarantees a graph even if every param is frozen
        out = self.model(inp)
        cls = int(out.argmax(1).item()) if cls is None else cls   # explicit None-check: `cls or ...` breaks for class 0
        self.model.zero_grad()
        g = torch.zeros_like(out); g[0, cls] = 1.0
        out.backward(gradient=g)
        w = self._grad.mean([2, 3], keepdim=True)
        cam = F.relu((w * self._act).sum(1, keepdim=True))
        cam = F.interpolate(cam, inp.shape[-2:], mode="bilinear", align_corners=False)
        cam = cam.squeeze().detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, cls

def get_gradcam_layer(model, backbone):
    if backbone.startswith("efficientnet"):
        return model.features[-1]
    return model.layer4[-1]  # resnet50

print("GradCAM defined.")


In [ ]:
# ── 9.2b) Visualize Grad-CAM overlays ────────────────────────────────────────────
N_SAMPLES = 6
gradcam = GradCAM(best_res["model"], get_gradcam_layer(best_res["model"], best_res["backbone"]))
rng3 = np.random.default_rng(SEED + 99)
sample_indices = rng3.choice(len(test_ds), N_SAMPLES, replace=False)

fig, axes = plt.subplots(N_SAMPLES, 3, figsize=(10, N_SAMPLES * 3.2))
for row, idx in enumerate(sample_indices):
    img_t, label = test_ds[int(idx)]
    inp = img_t.unsqueeze(0).to(device)
    cam, pred_cls = gradcam(inp)
    img_np = INV_NORM(img_t).permute(1, 2, 0).clamp(0, 1).numpy()

    gt_name = CLASS_NAMES[label].replace("_", " ")
    pred_name = CLASS_NAMES[pred_cls].replace("_", " ")
    tick = "correct" if pred_cls == label else "WRONG"

    axes[row, 0].imshow(img_np); axes[row, 0].set_title(f"GT: {gt_name}", fontsize=8); axes[row, 0].axis("off")
    axes[row, 1].imshow(cam, cmap="jet"); axes[row, 1].set_title("Grad-CAM", fontsize=8); axes[row, 1].axis("off")
    axes[row, 2].imshow(img_np); axes[row, 2].imshow(cam, cmap="jet", alpha=0.45)
    axes[row, 2].set_title(f"Pred: {pred_name} ({tick})", fontsize=8); axes[row, 2].axis("off")

plt.suptitle(f"Grad-CAM - {best_res['label']}", fontsize=11)
plt.tight_layout(); plt.show()


## 10) Discussion & Limitations

**Resolution mismatch.** EfficientNetV2-S's pretrained weights are natively 384×384; everything here was
trained at 224×224 to fit a 12-hour Kaggle budget across all 5 experiments. This is a deliberate,
documented deviation, not an oversight — and it isn't purely a compromise: the EfficientNetV2 paper itself
found that training at a smaller resolution than the eval resolution speeds up training substantially with
a small, often slightly *positive*, effect on accuracy (a FixRes-style effect). It does mean some of what
ImageNet pretraining learned at high resolution is under-used here, and this project doesn't isolate that
effect - a natural follow-up would be to fine-tune the best configuration at 384×384 and compare.

**Single run per configuration.** Every experiment was trained once, not averaged over multiple seeds.
The bootstrap confidence intervals in Section 8 quantify sampling uncertainty from the fixed-size test set
- they do *not* capture training-run-to-run variance. A different random initialization or data order
could plausibly move any given configuration's accuracy by some amount we haven't measured here. Treat the
point estimates as a first pass, not a final word.

**Compute-driven scope reductions vs. the original proposal.** Fewer fine-tune epochs (15 vs. 30, with
early stopping), `nn.DataParallel` instead of a full multi-process distributed setup, and 224×224 instead
of native resolution. All are documented, deliberate tradeoffs made to fit the available compute budget
(a single Kaggle 12-hour GPU session), not silent corner-cutting.

**Benchmark comparison.** The literature reports a wide range of Food-101 Top-1 numbers for nominally
similar transfer-learning setups - anywhere from the low 80s to high 90s - largely because training recipe,
resolution, and epoch count vary a lot between papers, and not all of them are rigorous about train/val/test
separation. We treat our own numbers as internally comparable across E0-E4 (same protocol, same
train/val/test split) rather than as directly comparable to any single external number.

**Noise framing.** Food-101's training-set noise, per Bossard et al.'s own description, is "mostly...
intense colors and sometimes wrong labels" - it is not purely a label-noise problem, and we haven't
attempted to separate the two effects. This is worth keeping in mind when interpreting why SCE/LS/CutMix
do or don't help: some of what they're compensating for may be visual noise rather than category noise.

**A code-level fragility worth noting for reproducibility.** The stratified train/val split relies on
torchvision's `Food101._labels`, an undocumented private attribute (with a slower fallback implemented in
case it's ever removed). If torchvision's internal implementation changes in a future release, that
fallback path is what keeps this notebook working.


## 11) Conclusion

*(Fill in after running all cells: which configuration won on the held-out test set, whether the delta over
the CE baseline was significant per the bootstrap test in Section 8, and what the confusion matrix / Grad-CAM
results suggest about where the model still struggles.)*

| Experiment | Change | Outcome |
|---|---|---|
| E0 ResNet-50 + CE | Architecture baseline | — |
| E1 EfficientNetV2-S + CE | Stronger backbone | Δ vs E0? (see Section 8e) |
| E2 EfficientNetV2-S + LS | Label smoothing ε=0.1 | Δ vs E1? |
| E3 EfficientNetV2-S + SCE | Symmetric loss α=0.1, β=1 | Δ vs E1? |
| E4 EfficientNetV2-S + CutMix | Patch mixing augmentation | Δ vs E1? |

### References
1. Bossard, L., Guillaumin, M., & Van Gool, L. (2014). Food-101 — Mining discriminative components with random forests. *ECCV*.
2. Tan, M., & Le, Q. (2021). EfficientNetV2: Smaller models and faster training. *ICML*.
3. Wang, Y., et al. (2019). Symmetric Cross Entropy for Robust Learning With Noisy Labels. *ICCV*.
4. Yun, S., et al. (2019). CutMix: Regularization Strategy to Train Strong Classifiers. *ICCV*.
5. Selvaraju, R. R., et al. (2017). Grad-CAM: Visual explanations from deep networks via gradient-based localization. *ICCV*.
6. Lee, K.-H., et al. (2018). CleanNet: Transfer Learning for Scalable Image Classifier Training with Label Noise. *CVPR*. (Food-101N, referenced in Section 2 to distinguish from Food-101.)
